##  Instalación de dependencias para el sistema RAG

In [24]:
#Instalacion de dependencias
!pip install langchain langchain-mistralai langchain-chroma chromadb sentence-transformers pandas tabulate


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Imports

In [25]:
# Imports de librerías
import os
import json
import pandas as pd
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import DataFrameLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

print(" Librerías cargadas correctamente")

 Librerías cargadas correctamente


## Configurar Mistral

In [26]:
# Configurar Mistral con dos modelos
# mistral-small: para el bot interactivo (rápido y económico)
# mistral-large: para el reporte ejecutivo (mayor precisión)
os.environ["MISTRAL_API_KEY"] = "ttlUH11iq9d1roD58pA3gOlq4DC0zPet"

llm_small = ChatMistralAI(
    model="mistral-small-latest",
    temperature=0.3
)

llm_large = ChatMistralAI(
    model="mistral-large-latest",
    temperature=0.3
)

# Test de ambos modelos para verificar que funcionan correctamente
test = llm_small.invoke("Respondé solo: ¿estás funcionando?")
print(test.content)
print(" mistral-small listo para el bot")
print(" mistral-large listo para el reporte ejecutivo")

Sí, estoy funcionando correctamente. ¿En qué puedo ayudarte?
 mistral-small listo para el bot
 mistral-large listo para el reporte ejecutivo


## Cargar dataset y métricas

In [27]:
# C Cargar dataset y métricas
# Cargamos tres fuentes de datos que generaron los agentes anteriores:
# 1. telco_limpio.csv     → dataset procesado por el Agente 1
# 2. Telco-Customer-Churn.csv → dataset original con valores legibles para el RAG
# 3. metricas.json        → resultados del entrenamiento del Agente 2

# Dataset procesado por el Agente 1 (valores escalados y codificados)
df = pd.read_csv("telco_limpio.csv")

# Dataset original con valores legibles en lenguaje natural
# Lo necesitamos porque el RAG debe entender "Month-to-month", "Fiber optic", etc.
# El dataset limpio tiene esos valores convertidos a números que Mistral no puede interpretar
df_original = pd.read_csv("Telco-Customer-Churn.csv")

# Métricas generadas por el Agente 2 (AUC, F1, modelo ganador, etc.)
with open("metricas.json", "r") as f:
    metricas = json.load(f)

print(f"Dataset limpio cargado:    {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Dataset original cargado:  {df_original.shape[0]} filas x {df_original.shape[1]} columnas")
print(f"Métricas cargadas:         {list(metricas.keys())}")

Dataset limpio cargado:    7043 filas x 20 columnas
Dataset original cargado:  7043 filas x 21 columnas
Métricas cargadas:         ['modelo_ganador', 'auc_rf', 'auc_lr', 'cv_rf_media', 'cv_lr_media', 'reporte_rf', 'reporte_lr']
